# Эксперименты: новые признаки, отбор, модели

В `baseline.ipynb` CatBoost на поведенческих признаках дал 0.77. Здесь проверяем, улучшат ли результат признаки для мобильных ботов, удаление дублей и другие модели.

In [1]:
import re
import sys
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

ROOT = Path.cwd().parent / 'bot_detection_challenge'
sys.path.append(str(ROOT))
from metric import precision_at_recall

SEED = 42
DATE_COLS = ['cookie_created_at', 'window_start_ts', 'window_end_ts']

train = pd.read_csv(ROOT / 'data/train.csv', parse_dates=DATE_COLS)
test = pd.read_csv(ROOT / 'data/test.csv', parse_dates=DATE_COLS)
events_raw = pd.read_csv(ROOT / 'data/events.csv.gz', parse_dates=['event_ts'])
meta = pd.concat([train, test]).set_index('cookie_id')
y = train.target.to_numpy()

## Данные, валидация, признаки

То же, что в `EDA.ipynb` и `baseline.ipynb`.

In [2]:
start = events_raw.cookie_id.map(meta.window_start_ts)
end = events_raw.cookie_id.map(meta.window_end_ts)
events = events_raw[(events_raw.event_ts >= start) & (events_raw.event_ts < end)]

events = events.drop_duplicates().drop(columns='eid')

PLATFORM_MAP = {'android': 'android', 'ios': 'ios', 'iphone': 'ios', 'web': 'web', 'desktop': 'web'}
events['platform'] = events.platform.str.lower().map(PLATFORM_MAP)

events = events.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

In [3]:
days = np.sort(train.window_start_ts.unique())
folds = []
for val_days in [days[6:8], days[8:10], days[10:12], days[12:14]]:
    folds.append((np.flatnonzero(train.window_start_ts < val_days[0]),
                  np.flatnonzero(train.window_start_ts.isin(val_days))))

testlike_fold = [(np.flatnonzero(train.window_start_ts < '2026-04-13'),
                  np.flatnonzero(train.window_start_ts >= '2026-04-13'))]

In [4]:
def base_features(events):
    counts = pd.crosstab(events.cookie_id, events.event_name)
    shares = counts.div(counts.sum(axis=1), axis=0)
    g = events.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'n_items': g.item_id.nunique(),
        'platform': g.platform.first(),
    })
    f['cookie_age_days'] = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400
    return f.join(counts.add_prefix('cnt_')).join(shares.add_prefix('share_'))


SESSION_GAP = 30 * 60


def timing_features(events):
    dt = events.groupby('cookie_id').event_ts.diff().dt.total_seconds()
    ev = events.assign(dt=dt, hour=events.event_ts.dt.hour)
    ev['dt_session'] = ev.dt.where(ev.dt < SESSION_GAP)
    ev['log_dt_session'] = np.log1p(ev.dt_session)
    g = ev.groupby('cookie_id')
    known = ev[ev.dt.notna()]

    f = pd.DataFrame({
        'span_hours': (g.event_ts.max() - g.event_ts.min()).dt.total_seconds() / 3600,
        'dt_min': g.dt.min(),
        'dt_q10': g.dt.quantile(0.1),
        'dt_median': g.dt.median(),
        'dts_median': g.dt_session.median(),
        'dts_mean': g.dt_session.mean(),
        'dts_std': g.dt_session.std(),
        'log_dts_std': g.log_dt_session.std(),
        'dts_nunique_ratio': g.dt_session.nunique() / g.dt_session.count(),
        'share_dt_0': (known.dt == 0).groupby(known.cookie_id).mean(),
        'share_dt_le10': (known.dt <= 10).groupby(known.cookie_id).mean(),
        'n_sessions': 1 + (ev.dt >= SESSION_GAP).groupby(ev.cookie_id).sum(),
        'n_active_hours': g.hour.nunique(),
        'share_night': (ev.hour < 6).groupby(ev.cookie_id).mean(),
    })
    f['dts_cv'] = f.dts_std / f.dts_mean
    f['events_per_session'] = g.size() / f.n_sessions
    f['events_per_active_hour'] = g.size() / f.n_active_hours
    return f


def search_features(events):
    s = events[events.event_name == 'search_results_view'].copy()
    g = s.groupby('cookie_id')
    s['same_query'] = s.search_query == g.search_query.shift()
    s['next_page'] = s.same_query & (s.search_page == g.search_page.shift() + 1)
    g = s.groupby('cookie_id')

    f = pd.DataFrame({
        'n_queries': g.search_query.nunique(),
        'page_max': g.search_page.max(),
        'page_mean': g.search_page.mean(),
        'share_page_gt1': (s.search_page > 1).groupby(s.cookie_id).mean(),
        'share_page_ge5': (s.search_page >= 5).groupby(s.cookie_id).mean(),
        'share_same_query': g.same_query.mean(),
        'share_next_page': g.next_page.mean(),
        'query_len_mean': s.search_query.str.len().groupby(s.cookie_id).mean(),
    })
    f['searches_per_query'] = g.size() / f.n_queries
    return f


def entropy(events, col):
    counts = events.dropna(subset=[col]).groupby(['cookie_id', col]).size()
    p = counts / counts.groupby(level=0).transform('sum')
    return (-p * np.log(p)).groupby(level=0).sum()


def diversity_features(events):
    g = events.groupby('cookie_id')
    items = events[events.item_id.notna()]
    gi = items.groupby('cookie_id')
    category_counts = events.groupby(['cookie_id', 'item_category']).size()
    is_pro = items.seller_type.eq('pro').astype(float).where(items.seller_type.notna())

    return pd.DataFrame({
        'n_categories': g.item_category.nunique(),
        'n_locations': g.item_location.nunique(),
        'category_entropy': entropy(events, 'item_category'),
        'location_entropy': entropy(events, 'item_location'),
        'top_category_share': category_counts.groupby(level=0).max() / category_counts.groupby(level=0).sum(),
        'item_events_per_item': gi.size() / gi.item_id.nunique(),
        'share_pro_seller': is_pro.groupby(items.cookie_id).mean(),
    })


SHORT_NAMES = {'search_results_view': 'srch', 'item_view': 'item', 'photo_swipe': 'photo', 'seller_page_view': 'seller'}
TRANSITIONS = ['srch>item', 'item>srch', 'item>item', 'srch>srch', 'item>photo', 'item>seller']


def sequence_features(events):
    name = events.event_name.map(SHORT_NAMES).fillna(events.event_name)
    prev = name.groupby(events.cookie_id).shift()
    pair = prev + '>' + name
    n_transitions = pair.notna().groupby(events.cookie_id).sum()

    f = pd.crosstab(events.cookie_id, pair).reindex(columns=TRANSITIONS, fill_value=0).div(n_transitions, axis=0)
    f.columns = ['tr_' + c.replace('>', '_to_') for c in f.columns]
    f['share_repeat_event'] = (prev == name).groupby(events.cookie_id).sum() / n_transitions
    return f


def pointer_features(events):
    web = events[events.platform == 'web']
    p = web.dropna(subset=['pointer_x'])
    step = np.hypot(p.groupby('cookie_id').pointer_x.diff(), p.groupby('cookie_id').pointer_y.diff())
    g = p.assign(step=step).groupby('cookie_id')

    return pd.DataFrame({
        'pointer_share': web.pointer_x.notna().groupby(web.cookie_id).mean(),
        'px_mean': g.pointer_x.mean(),
        'px_std': g.pointer_x.std(),
        'px_max': g.pointer_x.max(),
        'py_mean': g.pointer_y.mean(),
        'py_std': g.pointer_y.std(),
        'pointer_step_mean': g.step.mean(),
    })


SCRIPT_UA = re.compile(r'python-requests|curl|node-fetch|urllib|Scrapy|Go-http-client', re.I)
BROWSERS = ['HeadlessChrome', 'YaBrowser', 'Edg', 'OPR', 'Firefox', 'Chrome', 'Safari']
OS_NAMES = {'Android': 'android', 'iPhone': 'ios', 'Windows': 'windows', 'Mac OS': 'mac', 'Linux': 'linux'}


def ua_client(ua):
    if ua.startswith('Avito/'):
        return 'avito_app'
    if SCRIPT_UA.search(ua):
        return 'script'
    return next((b for b in BROWSERS if b in ua), 'other')


def ua_os(ua):
    return next((name for marker, name in OS_NAMES.items() if marker in ua), 'other')


def ua_features(events):
    g = events.groupby('cookie_id').user_agent
    main_ua = g.agg(lambda s: s.value_counts().index[0])
    return pd.DataFrame({
        'ua_client': main_ua.map(ua_client),
        'ua_os': main_ua.map(ua_os),
        'ua_version': main_ua.str.extract(r'(?:Avito|Chrome|Firefox|Version)/(\d+)')[0].astype(float),
        'n_user_agents': g.nunique(),
        'main_ua_share': g.agg(lambda s: s.value_counts(normalize=True).iloc[0]),
        'has_script_ua': g.agg(lambda s: s.str.contains(SCRIPT_UA).any()).astype(int),
    })

In [5]:
features = {
    'base': base_features(events),
    'timing': timing_features(events),
    'search': search_features(events),
    'diversity': diversity_features(events),
    'sequence': sequence_features(events),
    'pointer': pointer_features(events),
    'ua': ua_features(events),
}
GROUPS = list(features)


def assemble(groups, cookies=train):
    X = pd.concat([features[g] for g in groups], axis=1).reindex(cookies.cookie_id).reset_index(drop=True)
    cat_cols = X.select_dtypes('object').columns
    X[cat_cols] = X[cat_cols].fillna('missing')
    return X

## Шум от сида

Разница между вариантами будет небольшой, поэтому сначала смотрим, сколько шума даёт сам сид модели. Дальше предсказания усредняем по 3 сидам и считаем 95% бутстрап-интервал.

In [6]:
SEEDS = [42, 7, 2026]
CATBOOST_PARAMS = dict(iterations=800, learning_rate=0.03, depth=6, verbose=0, thread_count=-1, allow_writing_files=False)


def make_catboost(X, seed, **params):
    return CatBoostClassifier(**{**CATBOOST_PARAMS, **params}, random_seed=seed,
                              cat_features=X.select_dtypes('object').columns.tolist())


def oof_predict(make_model, X, folds=folds, seeds=SEEDS):
    oof = np.full(len(y), np.nan)
    for tr, va in folds:
        oof[va] = np.mean([make_model(X, s).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[va])[:, 1] for s in seeds], axis=0)
    return oof


def evaluate(oof, folds=folds):
    mask = ~np.isnan(oof)
    y_val, p_val = y[mask], oof[mask]
    rng = np.random.default_rng(SEED)
    boot = [precision_at_recall(y_val[i], p_val[i]) for i in rng.integers(0, len(y_val), (1000, len(y_val)))]
    return {
        'p_at_r70': round(precision_at_recall(y_val, p_val), 3),
        'ci95': np.percentile(boot, [2.5, 97.5]).round(3),
        'pr_auc': round(average_precision_score(y_val, p_val), 3),
        'per_fold': np.round([precision_at_recall(y[va], oof[va]) for _, va in folds], 3),
    }

In [7]:
X_m1 = assemble(GROUPS)
per_seed = {s: oof_predict(make_catboost, X_m1, seeds=[s]) for s in SEEDS}
oof_m1 = np.mean(list(per_seed.values()), axis=0)

pd.DataFrame({**{f'seed {s}': evaluate(o) for s, o in per_seed.items()},
              'среднее 3 сидов': evaluate(oof_m1)}).T

,p_at_r70,ci95,pr_auc,per_fold
seed 42,0.766,"[0.703, 0.809]",0.783,"[0.768, 0.789, 0.736, 0.75]"
seed 7,0.747,"[0.682, 0.792]",0.78,"[0.75, 0.738, 0.738, 0.725]"
seed 2026,0.736,"[0.677, 0.798]",0.781,"[0.75, 0.772, 0.703, 0.759]"
среднее 3 сидов,0.738,"[0.689, 0.805]",0.783,"[0.778, 0.774, 0.724, 0.733]"


Смена сида сдвигает P@R0.7 на ±0.015, а PR-AUC почти не меняется. Поэтому варианты сравниваем по среднему трёх сидов и смотрим также на PR-AUC.

## Мобильные боты

На android модель находит 57% ботов, на web — 77%: курсора на мобильных нет.

In [8]:
cookies = pd.DataFrame({
    'target': y,
    'platform': X_m1.platform,
    'client': X_m1.ua_client,
    'version': X_m1.ua_version,
})
mobile = cookies[cookies.platform != 'web']
display(mobile.groupby('client').target.agg(n='size', bot_rate='mean').round(3))
mobile[mobile.client == 'avito_app'].groupby('version').target.agg(n='size', bot_rate='mean').round(3)

,n,bot_rate
client,,
Chrome,2841,0.037
Safari,192,0.047
avito_app,2141,0.092


,n,bot_rate
version,,
118.0,643,0.058
120.0,700,0.090
122.0,798,0.122


В приложении Авито доля ботов в 2.5 раза выше, чем в мобильном браузере, и растёт с версией приложения.

Гипотезы:
- боты делают много действий за короткое время;
- мобильные боты выгружают контакты продавцов (такие сервисы упомянуты в условии).

In [9]:
def burst_features(events):
    """Всплески активности: сколько событий приходится на минуту, 10 минут, час."""
    g = events.groupby('cookie_id')
    per_min = events.groupby(['cookie_id', events.event_ts.dt.floor('min')]).size()
    per_10min = events.groupby(['cookie_id', events.event_ts.dt.floor('10min')]).size()
    per_hour = events.groupby(['cookie_id', events.event_ts.dt.floor('h')]).size()
    n_active_minutes = per_min.groupby(level=0).size()
    return pd.DataFrame({
        'events_per_active_min': g.size() / n_active_minutes,
        'max_per_min': per_min.groupby(level=0).max(),
        'max_per_10min': per_10min.groupby(level=0).max(),
        'max_per_hour': per_hour.groupby(level=0).max(),
        'n_active_minutes': n_active_minutes,
    })


def mobile_features(events):
    search = events[events.event_name == 'search_results_view']
    gs = search.groupby('cookie_id')
    n_item_views = (events.event_name == 'item_view').groupby(events.cookie_id).sum()
    return pd.DataFrame({
        'share_contact': events.event_name.str.startswith('contact').groupby(events.cookie_id).mean(),
        'is_app': (features['ua'].ua_client == 'avito_app').astype(int),
        'query_nunique_per_cat': gs.search_query.nunique() / gs.item_category.nunique().replace(0, np.nan),
        'items_per_search': n_item_views / gs.size(),
        'n_seller_types': events.groupby('cookie_id').seller_type.nunique(),
    })


features['burst'] = burst_features(events)
features['mobile'] = mobile_features(events)

new = assemble(['burst', 'mobile'])
is_mobile = (cookies.platform != 'web').to_numpy()
auc = {}
for col in new:
    v = new[col].to_numpy(dtype=float)
    ok = ~np.isnan(v)
    auc[col] = {'auc_all': roc_auc_score(y[ok], v[ok]),
                'auc_mobile': roc_auc_score(y[ok & is_mobile], v[ok & is_mobile])}
pd.DataFrame(auc).T.round(3)

,auc_all,auc_mobile
events_per_active_min,0.725,0.739
max_per_min,0.693,0.697
max_per_10min,0.693,0.684
max_per_hour,0.691,0.649
n_active_minutes,0.544,0.495
share_contact,0.503,0.647
is_app,0.514,0.617
query_nunique_per_cat,0.669,0.604
items_per_search,0.547,0.583
n_seller_types,0.567,0.559


Обе гипотезы подтвердились: `events_per_active_min` (событий на активную минуту) работает на всех куках, `share_contact` (доля контактных действий) — на мобильных.

In [10]:
feature_sets = {
    'M1: признаки из baseline': GROUPS,
    '+ всплески': GROUPS + ['burst'],
    '+ мобильные': GROUPS + ['mobile'],
    '+ всплески + мобильные': GROUPS + ['burst', 'mobile'],
}
set_results = {'M1: признаки из baseline': evaluate(oof_m1)}
for name, groups in list(feature_sets.items())[1:]:
    set_results[name] = evaluate(oof_predict(make_catboost, assemble(groups)))
pd.DataFrame(set_results).T

,p_at_r70,ci95,pr_auc,per_fold
M1: признаки из baseline,0.738,"[0.689, 0.805]",0.783,"[0.778, 0.774, 0.724, 0.733]"
+ всплески,0.735,"[0.682, 0.803]",0.785,"[0.792, 0.748, 0.724, 0.725]"
+ мобильные,0.76,"[0.693, 0.811]",0.788,"[0.8, 0.762, 0.742, 0.759]"
+ всплески + мобильные,0.742,"[0.675, 0.812]",0.789,"[0.771, 0.762, 0.73, 0.767]"


Все варианты лежат внутри интервалов друг друга: новые признаки во многом повторяют уже имеющиеся.

## Дрейф и дубли

Adversarial validation: модель учится отличать train от test. ROC-AUC около 0.5 означает, что дрейфа нет.

In [11]:
ALL_GROUPS = GROUPS + ['burst', 'mobile']
X_adv = assemble(ALL_GROUPS, pd.concat([train, test]))
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))].astype(int)

adv_oof = np.zeros(len(X_adv))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=SEED).split(X_adv, is_test):
    model = make_catboost(X_adv, SEED, iterations=300)
    model.fit(X_adv.iloc[tr], is_test[tr])
    adv_oof[va] = model.predict_proba(X_adv.iloc[va])[:, 1]

round(roc_auc_score(is_test, adv_oof), 3)

0.5

Теперь удалим дублирующиеся признаки. Используем для этого корреляцию Спирмена, чтобы учесть все монотонные связи.

In [12]:
X_all = assemble(ALL_GROUPS)
corr = X_all.select_dtypes('number').corr(method='spearman').abs()
pairs = [(a, b, round(corr.loc[a, b], 3)) for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:]
         if corr.loc[a, b] > 0.95]
pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'spearman'])

,feature_1,feature_2,spearman
0,n_events,n_items,0.960
1,cnt_contact_chat_open,share_contact_chat_open,0.985
2,cnt_contact_message_sent,share_contact_message_sent,0.995
3,cnt_contact_phone_show,share_contact_phone_show,0.961
4,cnt_login,share_login,0.979
5,cnt_search_results_view,n_queries,0.953
6,span_hours,n_sessions,0.958
7,n_sessions,n_active_hours,0.951
8,n_categories,category_entropy,0.965
9,category_entropy,top_category_share,0.970


Дрейфа нет.

Из каждой пары с корреляцией Спирмена выше 0.95 оставляем один признак.

In [13]:
REDUNDANT = ['n_items', 'cnt_contact_chat_open', 'cnt_contact_message_sent', 'cnt_contact_phone_show', 'cnt_login',
             'n_queries', 'n_sessions', 'category_entropy', 'main_ua_share']
X_m2 = X_all.drop(columns=REDUNDANT)
set_results['M2: + всплески + мобильные − дубли'] = evaluate(oof_predict(make_catboost, X_m2))
print('признаков в M2:', X_m2.shape[1])
pd.DataFrame(set_results).T

признаков в M2: 76


,p_at_r70,ci95,pr_auc,per_fold
M1: признаки из baseline,0.738,"[0.689, 0.805]",0.783,"[0.778, 0.774, 0.724, 0.733]"
+ всплески,0.735,"[0.682, 0.803]",0.785,"[0.792, 0.748, 0.724, 0.725]"
+ мобильные,0.76,"[0.693, 0.811]",0.788,"[0.8, 0.762, 0.742, 0.759]"
+ всплески + мобильные,0.742,"[0.675, 0.812]",0.789,"[0.771, 0.762, 0.73, 0.767]"
M2: + всплески + мобильные − дубли,0.755,"[0.673, 0.821]",0.789,"[0.766, 0.76, 0.724, 0.736]"


## Модели

Сравниваем на M2. Блендинг — среднее рангов: у моделей разные шкалы вероятностей, а метрике важен только порядок.

In [14]:
X_m2_lgb = X_m2.copy()
for col in X_m2_lgb.select_dtypes('object'):
    X_m2_lgb[col] = X_m2_lgb[col].astype('category')


def make_catboost_slow(X, seed):
    return make_catboost(X, seed, learning_rate=0.015, iterations=1600)


def make_lightgbm(X, seed):
    return lgb.LGBMClassifier(n_estimators=600, learning_rate=0.02, num_leaves=15, min_child_samples=30,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
                              random_state=seed, verbose=-1, n_jobs=-1)


def rank_blend(a, b):
    mask = ~np.isnan(a)
    blend = np.full(len(a), np.nan)
    blend[mask] = (rankdata(a[mask]) + rankdata(b[mask])) / (2 * mask.sum())
    return blend


oof_cb = oof_predict(make_catboost_slow, X_m2)
oof_lgb = oof_predict(make_lightgbm, X_m2_lgb)
pd.DataFrame({
    'CatBoost': set_results['M2: + всплески + мобильные − дубли'],
    'CatBoost, lr=0.015, 1600 деревьев': evaluate(oof_cb),
    'LightGBM': evaluate(oof_lgb),
    'Бленд CatBoost + LightGBM': evaluate(rank_blend(oof_cb, oof_lgb)),
}).T

,p_at_r70,ci95,pr_auc,per_fold
CatBoost,0.755,"[0.673, 0.821]",0.789,"[0.766, 0.76, 0.724, 0.736]"
"CatBoost, lr=0.015, 1600 деревьев",0.756,"[0.674, 0.825]",0.79,"[0.764, 0.754, 0.742, 0.726]"
LightGBM,0.785,"[0.676, 0.841]",0.787,"[0.832, 0.754, 0.783, 0.759]"
Бленд CatBoost + LightGBM,0.767,"[0.69, 0.841]",0.792,"[0.8, 0.76, 0.774, 0.736]"


In [15]:
tl_cb = oof_predict(make_catboost_slow, X_m2, folds=testlike_fold)
tl_lgb = oof_predict(make_lightgbm, X_m2_lgb, folds=testlike_fold)
pd.DataFrame({
    'CatBoost, lr=0.015': evaluate(tl_cb, testlike_fold),
    'LightGBM': evaluate(tl_lgb, testlike_fold),
    'Бленд': evaluate(rank_blend(tl_cb, tl_lgb), testlike_fold),
}).T

,p_at_r70,ci95,pr_auc,per_fold
"CatBoost, lr=0.015",0.711,"[0.64, 0.801]",0.785,[0.711]
LightGBM,0.749,"[0.65, 0.84]",0.784,[0.749]
Бленд,0.727,"[0.664, 0.823]",0.788,[0.727]


LightGBM выше по P@R0.7, бленд — по PR-AUC, интервалы почти совпадают. Берём бленд: у него выше нижняя граница интервала, а в итоговом решении один сид, и бленд двух алгоритмов меньше от него зависит.

## Итог

- Модели и наборы признаков упираются в одно плато, разница между ними меньше доверительного интервала.
- Другие гиперпараметры, веса классов и отдельные модели для web и мобильных тоже не дали прироста.
- Признаки для мобильных ботов работают по отдельности, но recall на мобильных почти не вырос.

В `solution.ipynb` берём блендинг на M2.